# VoicePrint — Fine-Tune Mistral 7B Instruct v0.3 for AI→Human Style Transfer
---
**Phase 2:** Train a small, fast model that rewrites AI-generated text to sound human.

**Architecture:** Mistral 7B Instruct v0.3 + QLoRA via Unsloth  
**Dataset:** ~76K paired AI→Human examples (HC3 + CMU Human-AI Parallel)  
**Output:** LoRA adapter → merged GGUF for local CPU inference

Runtime: ~55 minutes on a free Colab T4 GPU.

Run cells in order. Cells 6 (synthetic pair gen) is optional — skip it for faster runs.

In [ ]:
# @title 1. Install dependencies
import subprocess, sys, os, json

print("Installing Unsloth + deps...")
subprocess.run([sys.executable, "-m", "pip", "install", "-qU",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "trl", "peft", "accelerate", "bitsandbytes",
    "xformers", "triton",
    "datasets", "huggingface_hub",
    "sentencepiece", "protobuf",
], check=True)
print("Done.")

In [ ]:
# @title 2. Embed Medium articles for style-specific training
MEDIUM_CHUNKS = [
    "Your first agent might ship with a 500-token system prompt and two tools, but those numbers usually balloon fast. Just to illustrate, the leaked Claude system prompt ran around 24,000 tokens, and OpenClaw users have reported more than 150,000 input tokens sent to Gemini 3.1 Pro for 29 tokens of output on the first turn. An unoptimized agent that runs at 100 messages a day at 166K input tokens can cost around $996 a month on Gemini 3.1 Pro and roughly $2,490 on Claude Opus 4.6. There are tricks to keep these costs down, closer to $50 and $100 a month.",
    "I encourage subscribing to Medium, but if you can't right now, you can still read this story on my website www.appliedtom.com Let's do a pulse check! Are you psychologically safe at work? It's a question that's been getting more attention lately, and for good reason. The way we feel at work affects everything — our productivity, our creativity, and honestly, our will to show up every day.",
    "Time may change me, but I can't trace time. What 50 Years of Photography Has Taught Me About Seeing — Lessons about observation, ordinary moments, and the photographs we almost miss. The first thing you learn when you carry a camera for decades is that most of the great photographs were almost not taken. They happened because someone was paying attention when it would have been easier to look away.",
    "This 2800-word essay (a 12-minute read) is about how to survive inside the AI revolution in software development, without succumbing to the fear that swirls around all of us. It explains some lessons I learned hiking up difficult mountain trails that are useful for wrestling with the coding agents. They apply to all knowledge workers, I think. The AI revolution in software isn't coming — it's here.",
    "It's been a full decade since I started my blog Go Do, and writing consistently hasn't gotten any easier. In fact, it feels harder than ever before. When I started, I had this romantic idea that writing would get easier with practice. That the words would flow more freely, that I'd find my rhythm and ride it. Ten years in, I can tell you that's not exactly how it works.",
    "So I wanted to go through a few design principles people usually consider when building. We'll go through how prompt caching works and why it's a quick win, semantic caching, lazy-loading tools and MCPs, routing and cascading, delegating to subagents, and a bit on what it saves to keep the context clean. I am including interactive graphs throughout this article that help you visualize the cost savings each principle can get you based on the amount of tokens you are using.",
    "Yes, I am obviously staying real throughout, every saving comes with trade-offs. Prompt caching saves money but adds complexity. Semantic caching is powerful but you need to tune the similarity thresholds. Lazy-loading tools keeps your context clean but adds latency on first call. Routing sounds great until you realize you're maintaining multiple prompt templates.",
    "Being psychologically safe means you can speak up without fear of punishment or humiliation. It means you can admit mistakes, ask questions, or challenge ideas without worrying about your job security or your reputation. When you have psychological safety, you don't spend half your mental energy on office politics or reading between the lines of every email.",
    "The research is pretty clear on this. Google's Project Aristotle found that psychological safety was the number one predictor of team effectiveness. Not IQ, not experience, not even the talent on the team. It was whether people felt safe enough to take risks and be vulnerable with each other.",
    "Photography taught me that the best images aren't about the subject at all. They're about the relationship between the photographer and the subject. That relationship is built on patience, on quiet observation, on being present long enough for the world to reveal itself. You can't rush a good photograph any more than you can rush a good conversation.",
    "The magic happens in the ordinary moments, the ones we usually walk past. A shaft of light hitting a kitchen table. The way dust motes dance in afternoon sun. A stranger's expression caught for a split second. These are the photographs that matter, the ones that tell us something about being alive.",
    "The AI revolution in software isn't coming — it's here. And if you're a software developer, you're probably feeling a mix of excitement and existential dread. The excitement is obvious: these tools are genuinely incredible. The dread is more complicated. It's the quiet fear that maybe, just maybe, we're training our own replacements.",
    "Here's what I've learned from hiking: the trail doesn't get easier, you just get stronger. And sometimes, the trail doesn't get easier AND you don't get stronger — you just learn to carry the weight differently. That's where we are with AI in software development. The weight isn't going away. We just need to learn to carry it differently.",
    "The hardest lesson about blogging for a decade is that nobody owes you their attention. You have to earn it, every single time. And even when you earn it, they might not stay. Readers come and go, algorithms change, platforms rise and fall. The only thing you can control is whether you keep showing up and writing the next post.",
    "What kept me going wasn't success — because honestly, by most metrics, this blog has been a mediocre endeavor. What kept me going was the occasional email from someone who said something I wrote mattered to them. One sentence in one post that landed at exactly the right moment in someone's life. You can't plan for those moments. You can only stay in the game long enough for them to happen.",
]
print(f"Loaded {len(MEDIUM_CHUNKS)} Medium article chunks for style-specific training")

In [ ]:
# @title 3. Upload & load local processed data
import json, os
from google.colab import files

print("=" * 60)
print("Upload your local processed data files")
print("Using: train.jsonl + val.jsonl")
print("=" * 60)
print("\nStep 1: Select train.jsonl from VoicePrint/data/processed/")
print("Step 2: Select val.jsonl from the same folder")
print("\n--- Upload train.jsonl ---\n")
uploaded_train = files.upload()
train_file = list(uploaded_train.keys())[0]
print(f"\nGot {train_file} ({os.path.getsize(train_file)/1024**2:.0f} MB)")

print("\n--- Upload val.jsonl ---\n")
uploaded_val = files.upload()
val_file = list(uploaded_val.keys())[0]
print(f"Got {val_file} ({os.path.getsize(val_file)/1024**2:.0f} MB)")

# Load training pairs
train_source, train_target = [], []
print("\nLoading train.jsonl...")
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        src = d.get("source_text", "").strip()
        tgt = d.get("target_text", "").strip()
        if src and tgt and len(src) > 20 and len(tgt) > 20:
            train_source.append(src)
            train_target.append(tgt)
print(f"  Loaded {len(train_source)} training pairs")

val_source, val_target = [], []
print("Loading val.jsonl...")
with open(val_file, 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        src = d.get("source_text", "").strip()
        tgt = d.get("target_text", "").strip()
        if src and tgt and len(src) > 20 and len(tgt) > 20:
            val_source.append(src)
            val_target.append(tgt)
print(f"  Loaded {len(val_source)} validation pairs")

# Data sources used: HC3 (finance, medicine, open_qa, reddit_eli5, wiki_csai)
# + CMU Human-AI Parallel (GPT-4o, Llama-3, etc.)
# + synthetic pairs from your Medium articles (added in Cell 6)
#
# Note: ai2human (54K human sentences) is already used in the local
# pipeline as style references for the style scorer, not as Seq2Seq
# training pairs.

print(f"\nFinal: {len(train_source)} train + {len(val_source)} val = {len(train_source)+len(val_source)} total")

In [ ]:
# @title 4. Data quality check
print("=== Training data sample ===")
for i, (s, t) in enumerate(zip(train_source[:3], train_target[:3])):
    print(f"\n[Pair {i+1}]")
    print(f"  SOURCE ({len(s)} chars): {s[:120]}...")
    print(f"  TARGET ({len(t)} chars): {t[:120]}...")

print(f"\n=== Stats ===")
print(f"Train pairs : {len(train_source)}")
print(f"Val   pairs : {len(val_source)}")

# Basic length sanity
src_lens = [len(s) for s in train_source]
tgt_lens = [len(t) for t in train_target]
print(f"Source len  : min={min(src_lens)}  max={max(src_lens)}  avg={sum(src_lens)//len(src_lens)}")
print(f"Target len  : min={min(tgt_lens)}  max={max(tgt_lens)}  avg={sum(tgt_lens)//len(tgt_lens)}")

In [ ]:
# @title 5. Load Mistral 7B Instruct v0.3 with Unsloth (4-bit)
# Using the pre-quantised Unsloth build for faster loading and lower VRAM.
# The Unsloth variant is functionally identical to:
#   mistralai/Mistral-7B-Instruct-v0.3
# but ships BNB 4-bit weights pre-baked, saving ~10 min of quantisation
# on the free T4. VRAM usage: ~4.0 GB (vs ~14 GB for bf16).
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
max_seq_length = 2048

print(f"Loading {model_name} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# @title 6. (Optional) Generate synthetic pairs from Medium articles
GENERATE_SYNTHETIC = True   # Set to False to skip (~15 min)

if GENERATE_SYNTHETIC and MEDIUM_CHUNKS:
    print(f"Generating synthetic AI->Human pairs from {len(MEDIUM_CHUNKS)} chunks...")
    FastLanguageModel.for_inference(model)
    prompt_template = "Rewrite this to sound like an AI wrote it:\n{text}"
    for i, human_chunk in enumerate(MEDIUM_CHUNKS):
        prompt = prompt_template.format(text=human_chunk)
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt").to("cuda")
        outputs = model.generate(input_ids=inputs, max_new_tokens=512,
            temperature=1.0, top_p=0.9, do_sample=True)
        ai_text = tokenizer.decode(outputs[0][inputs.shape[1]:],
            skip_special_tokens=True).strip()
        if ai_text and len(ai_text) > 20:
            train_source.append(ai_text)
            train_target.append(human_chunk)
        if (i + 1) % 5 == 0:
            print(f"  Generated {i+1}/{len(MEDIUM_CHUNKS)}...")
    # Must call for_training() to restore Unsloth's training patches
    # after for_inference(). model.train() only sets dropout mode and
    # does NOT restore the patched forward pass.
    FastLanguageModel.for_training(model)
    print(f"Added synthetic pairs. Total: {len(train_source)}")
else:
    print("Skipping synthetic pair generation.")
    print(f"Training with {len(train_source)} source samples.")

In [ ]:
# @title 7. Configure LoRA
# Mistral 7B uses the same attention + MLP projection names as Llama,
# so the target_modules list is identical.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {model.get_nb_trainable_parameters()[0]:,}")
print(f"Effective batch size: 4 (per device) x 4 (grad accum) = 16")

In [ ]:
# @title 8. Build Mistral Instruct dataset
# Mistral Instruct v0.3 uses the [INST]...[/INST] chat template.
# We use tokenizer.apply_chat_template() to format pairs correctly —
# this handles BOS/EOS tokens and template quirks automatically.
# Do NOT use the old ChatML format here; it trains the model on the
# wrong tokens and hurts output naturalness.
from datasets import Dataset

def format_mistral(source: str, target: str) -> str:
    """Format as Mistral Instruct for causal LM training.

    Produces:
        [INST] Rewrite this to sound human:
        <source> [/INST] <target></s>
    """
    messages = [
        {"role": "user",      "content": f"Rewrite this to sound human:\n{source}"},
        {"role": "assistant", "content": target},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

train_formatted = [format_mistral(s, t) for s, t in zip(train_source, train_target)]
val_formatted   = [format_mistral(s, t) for s, t in zip(val_source,   val_target)]

train_ds = Dataset.from_dict({"text": train_formatted})
val_ds   = Dataset.from_dict({"text": val_formatted})

print(f"Dataset created: {len(train_ds)} train | {len(val_ds)} val")
print("--- Example (train[0], first 300 chars) ---")
print(train_ds[0]['text'][:300])
print("...")

In [ ]:
# @title 9. Train!
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        eval_steps=100,
        save_steps=200,
        eval_strategy="steps",
        output_dir="mistral-humanizer-lora",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)
print("Starting training (~55 min on T4)...")
trainer_stats = trainer.train()
print(f"\nTraining complete! Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# @title 10. Save LoRA adapter
import shutil
model.save_pretrained("mistral-humanizer-lora")
tokenizer.save_pretrained("mistral-humanizer-lora")
print("LoRA adapter saved")
shutil.make_archive("mistral-humanizer-lora", "zip", "mistral-humanizer-lora")
print("Zipped: mistral-humanizer-lora.zip")

In [ ]:
# @title 11. Merge LoRA into base model
import shutil
import os

if not os.path.exists("mistral-humanizer-lora"):
    raise FileNotFoundError(
        "LoRA adapter not found. Run cell 10 (Save LoRA adapter) first."
    )

print("Merging LoRA adapter...")
model.save_pretrained_merged(
    "mistral-humanizer-merged", tokenizer,
    save_method="merged_16bit")
print("Merged model saved")
shutil.make_archive("mistral-humanizer-merged", "zip", "mistral-humanizer-merged")
print("Zipped: mistral-humanizer-merged.zip")

In [ ]:
# @title 12. Convert to GGUF
import subprocess, sys
import os

if not os.path.exists("mistral-humanizer-merged"):
    raise FileNotFoundError(
        "Merged model not found. Run cell 11 (Merge LoRA) first."
    )

subprocess.run(["git", "clone", "--depth", "1",
    "https://github.com/ggerganov/llama.cpp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "sentencepiece", "protobuf"], check=True)
subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py",
    "mistral-humanizer-merged",
    "--outfile", "mistral-7b-humanizer.gguf",
    "--outtype", "q8_0"], check=True)
print("GGUF conversion complete: mistral-7b-humanizer.gguf")

In [ ]:
# @title 13. Download everything
from google.colab import files
import os
print("Files ready for download:")
for f in ["mistral-humanizer-lora.zip", "mistral-humanizer-merged.zip", "mistral-7b-humanizer.gguf"]:
    if os.path.exists(f):
        mb = os.path.getsize(f) / 1024**2
        print(f"  {f:45s} {mb:.1f} MB")
print("\n-> Place the GGUF at: VoicePrint/models/humanizer/mistral-7b-humanizer.gguf")
print("\n-> Uncomment to download the GGUF (6-8 GB):")
# files.download("mistral-7b-humanizer.gguf")

In [ ]:
# @title 14. Quick sanity check
import os, subprocess, sys

if os.path.exists("mistral-7b-humanizer.gguf"):
    print("Testing GGUF inference...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "llama-cpp-python"], check=True)
    from llama_cpp import Llama
    # n_ctx=2048 matches training max_seq_length
    llm = Llama(model_path="mistral-7b-humanizer.gguf", n_ctx=2048, verbose=False)
    resp = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "You rewrite AI-generated text to sound like it was "
                    "written by a human. Use contractions, vary sentence "
                    "structure, and avoid polished uniformity."
                ),
            },
            {
                "role": "user",
                "content": "Rewrite this to sound human: Furthermore, the implementation of comprehensive security protocols is essential for safeguarding organizational data assets against potential cyber threats.",
            },
        ],
        temperature=0.8,
        max_tokens=200,
    )
    print("\n=== Output ===")
    print(resp["choices"][0]["message"]["content"])
else:
    print("GGUF not found — run cells 11-12 first")